
# Sentinel‑2 Panel Cropper

This notebook scans a folder containing the automatically generated *panel* mosaics  
(e.g. `S2A_21WXU_20170219_0_L1C_20170219T153251_panel.png`) and cuts the six square
views:

| Row 1 | Row 2 |
|-------|-------|
| RGB   | Solid ice |
| Cloud | Light ice |
| Land  | Overlay |

Each crop is saved as an individual JPG named  
`YYYY‑MM‑DD_<layer>.jpg` in **`public/data/panels‑crops/`** so the website can load
them directly by date.

**How to run**

1. Put all `*_panel.png` files in `input_panels/` (or change `SRC_DIR` below).  
2. Select *Run All*.
3. Commit / deploy the resulting `public/data/panels‑crops/` folder.



In [1]:
# ⬇️ Install Pillow once (skip if already present)
# !pip install pillow

In [3]:

from pathlib import Path
import re
from PIL import Image

# ------- CONFIG ----------------------------------------------------
SRC_DIR = Path("out/imgtestNEW")                # where *_panel.png files live
DST_DIR = Path("out/panels-crops")    # where cropped tiles go
DST_DIR.mkdir(parents=True, exist_ok=True)

ORDER = [
    "rgb", "cloud", "land",
    "solid", "light", "overlay"
]
DATE_RX = re.compile(r"(\d{4})(\d{2})(\d{2})")  # 8‑digit date in filename


In [ ]:

def date_from_name(fname: str) -> str | None:
    """Extract YYYY‑MM‑DD from any filename containing 8 consecutive digits."""
    m = DATE_RX.search(fname)
    return f"{m[1]}-{m[2]}-{m[3]}" if m else None

def process_panels(src: Path = SRC_DIR, dst: Path = DST_DIR) -> None:
    processed = 0
    for fp in src.glob("*panel.png"):
        date = date_from_name(fp.name)
        if not date:
            print(f"skip {fp.name} – no date found")
            continue

        im = Image.open(fp)
        W, H = im.size
        cw, ch = W // 3, H // 2        # cell width/height
        side = min(cw, ch)             # enforce square

        for idx, tag in enumerate(ORDER):
            col = idx % 3
            row = idx // 3
            left = col * cw
            top  = row * ch
            crop = im.crop((left, top, left + side, top + side))

            crop = im.crop((left, top, left + side, top + side))

            # ↓ add this guard before saving
            if crop.mode in ("RGBA", "P"):
                crop = crop.convert("RGB")
            out  = dst / f"{date}_{tag}.jpg"
            crop.save(out, quality=92)
        processed += 1
    print(f"Done – {processed} mosaics processed, crops in {dst}")
    


In [5]:
process_panels()

OSError: cannot write mode RGBA as JPEG